In [ ]:
# Imports
import time

# Dependencies
import pandas						as pd

# Library
from api_builder.fsm				import fsm
from api_builder.fsm				import menus
from api_builder.fsm.states			import State
from api_builder.api_request		import header
from cards.fsm						import CardFSM
from api_builder.fsm.query_builder	import make_query
from api_builder.fsm.url_parser		import parse_links
from bs4							import BeautifulSoup
from api_builder.fsm.fetch			import fetch_routine
from parsers.vanguard_parser		import VanguardParser
from utils.utils					import construct_rules
from data.vanguard_data				import VanguardStorage
from pipeline.builder				import VanguardPipeline
from classifier.vanguard_classifier	import VanguardClassifier
from api_builder.fsm.scrap			import main_scrap_routine
from api_builder.vanguard_api_build	import MediaWikiAPI, VanguardScrapper


In [ ]:
# Url Parser
web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardParser(),
	VanguardStorage(),
	VanguardScrapper(web),
	VanguardClassifier()
)
await pipeline.scrapper.api.init_session()
state_machine = fsm.FSMContext()
state = State.ENTRY_POINT
while (state != State.END):
	if (state == State.ENTRY_POINT):
		state = menus.entry_point(state_machine)
	elif (state == State.SELECT_MAIN_CATEGORY):
		state = menus.select_category(state_machine)
	elif (state == State.SELECT_SUBCATEGORY):
		state = menus.select_subcategory(state_machine)
	elif (state == State.BUILD_QUERY):
		state = make_query(state_machine)
	elif (state == State.FETCH):
		state = await fetch_routine(state_machine, pipeline)
	elif (state == State.PARSE):
		pipeline.classifier._define_rules(construct_rules(state_machine.main_category.capitalize()))
		state = parse_links(state_machine, pipeline)
		card_fsm = CardFSM(state_machine)
		state = State.END
time.sleep(4)

In [ ]:
from data.check_data_base import build_set_path

for block in ["LB", "LL", "G", "V", "D", "DZ"]:
	consult = pipeline.parser.make_consults(getattr(pipeline.storage, block.lower()))
	print(consult)
	for tpl in consult.values():
		print(f"voy a pasar esta consulta: {tpl}")
		time.sleep(6)
		api_result = await pipeline.scrapper.api.get(
			params=tpl,
			headers=header
		)
		print(api_result)
		wikitex = pipeline.scrapper.obtain_wikitex(api_result)
		data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
		infobox = pipeline.parser.infobox(wikitex)
		is_d = False
		is_deck = False
		if (block in ["D", "DZ"]):
			is_d = True
		rows = pipeline.storage.prepare_data(data, 6, is_d=is_d, is_deck=is_deck, infobox=infobox)
		df = pd.DataFrame(rows, columns=[
			"CardNo", "Name", "Grade", "Faction",
			"FactionType", "Type", "Rarity", "Release"
		])
		set_number = pipeline.classifier.obtain_set_number(
			data[0]
		)
		path = build_set_path(
			category="boosters",
			set_type="main",
			block=block,
			set_number=set_number
		)
		path.parent.mkdir(
			parents=True,
			exist_ok=True
		)
		df.to_parquet(path)

		print(data)

In [ ]:
db = pd.read_parquet("database/boosters/booster sets/d/set_09.parquet")
db.loc[db["Code"] == "None"]

In [ ]:
db[110:130]

In [ ]:
pipeline.storage.d

In [ ]:
dz_consults = pipeline.parser.make_consults(pipeline.storage.d, "consult")
api_answer = await pipeline.scrapper.api.get(dz_consults[8], headers=header)

In [ ]:
soup = BeautifulSoup(api_answer["parse"]["text"]["*"], "html.parser")

In [ ]:
wikitex = pipeline.scrapper.obtain_wikitex(api_answer)
data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
infobox = pipeline.parser.infobox(wikitex)

In [ ]:
wiki = pipeline.scrapper.obtain_links(api_answer)
pipeline.parser.clean_trash_from_set(state_machine.data["page"], wiki, 4, reverse=True)

In [ ]:
param = {
    "action": "parse",
    "page": "V Booster Set 01: Unite! Team Q4",
    "prop": "links",
    "format": "json"
}

In [ ]:
await main_scrap_routine(card_fsm, pipeline)

In [ ]:
api_result = await pipeline.scrapper.api.get(
	params=dz_consults[0],
	headers=header
)